# Validation Test Evaluation (mesh study) - Capillary Rise (benchmark test case within SFB 1194)

This notebook and the corresponding startUp and simulation setup notebook (CapillaryRise_SFB1194_Run.ipynb, CapillaryRise_SFB1194_RunStartUp.ipynb) are part of published results (cf. 4.5) found in *A comparative study of transient capillary rise using direct numerical simulations* (https://doi.org/10.1016/j.apm.2020.04.020). 

In [ ]:
#r "BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Release/net8.0/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

MPI Rank 0: Value for OMP_PLACES: 
MPI Rank 0: Value for OMP_PROC_BIND: 
Number of CPUs in system: 24
not defined: OMP_NUM_THREADS
System reports 24 CPUs, 1 MPI ranks on current node.
Failed to determine user wish for number of threads; trying to use all! System reports 24 CPUs, will use all but 2 for BoSSS (22 total, 22 per MPI rank, MPI ranks on current node is 1).
Finally, setting number of OpenMP and Parallel Task Library threads to 22
CCP_AFFINITY not set
R0: reserved CPUs: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23], C# reports mask FFFFFF
R0: reserved CPUs for this MPI rank: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23], on entire SMP: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23], disjount CPU affinities? False
MpiJobOwnsEntireComputer = True, RnkJobOwnsEntireComputer = True
R0: using CPUs [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23] for OpenMP.
R0: TPL thread pinning: False, OMP thread pinning: False
R0

Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile() in C:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 263
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in C:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 105
   at Submission#73.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [ ]:
BoSSSshell.WorkflowMgm.Init("CapillaryRisePaper");

Opening existing database 'C:\BoSSS-localJobs\CapillaryRisePaper_OnJenkins'.


In [75]:
using System.IO;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

In [76]:
bool runShortTests = true; // set to 'false' to run all test cases for the whole period of time (up tp 0.7), otherwise they will run only up to 0.1

## Sessions

In [ ]:
static var sessions = BoSSSshell.WorkflowMgm.Sessions.ToList();
//static var sessions = databases.Pick(0).Sessions;
sessions

#0: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh1_startUp	11/24/2025 3:07:56 PM	243a4381...
#1: CapillaryRisePaper	CapillaryRise_Omega100_OmegaStudy_mesh8_startUp	11/24/2025 3:09:20 PM	e34da697...
#2: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh2_startUp	11/24/2025 3:08:05 PM	ca56d38d...
#3: CapillaryRisePaper	CapillaryRise_Omega10_OmegaStudy_mesh8_startUp	11/24/2025 3:09:09 PM	6c1e7c74...
#4: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh4_startUp	11/24/2025 3:08:13 PM	0d08da88...
#5: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh8_startUp*	11/24/2025 3:08:23 PM	52e81583...
#6: CapillaryRisePaper	CapillaryRise_Omega05_OmegaStudy_mesh8_AMR1_startUp*	11/24/2025 3:08:51 PM	89838821...
#7: CapillaryRisePaper	CapillaryRise_Omega01_OmegaStudy_mesh8_AMR1_startUp*	12/2/2025 2:06:56 PM	f072f08d...
#8: CapillaryRisePaper	CapillaryRise_Omega1_OmegaStudy_mesh8_startUp	11/24/2025 3:09:00 PM	0e9829d9...
#9: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh16

## Convergence studies

In [78]:
string[] studieS = { "effectiveSlip", "resolvedSlipR5", "resolvedSlipR50" };
string[] mesheS = { "mesh1", "mesh2", "mesh4", "mesh8", "mesh16", "mesh8_AMR1", "mesh8_AMR2" };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Nov26_000000.CapillaryRisePaper");

Opening existing database '\\fdygitrunner\ValidationTests\databases\bkup-2025Nov26_000000.CapillaryRisePaper'.


In [80]:
List<ISessionInfo> evalSessOrigin = new List<ISessionInfo>();

In [81]:
foreach (string study in studieS) {
foreach (string mesh in mesheS) {

    string studyName = $"CapillaryRise_Omega1_{study}_meshStudy_{mesh}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));

    if (mesh == "mesh1")
        studySess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName) && !sess.Name.Contains("mesh16"));
    if (mesh == "mesh8")
        studySess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName) && !sess.Name.Contains("_AMR"));

    int studyCount = studySess.Count();
    if (studyCount == 0) {
        Console.WriteLine("Not found in orignDB!");
    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSessOrigin.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSessOrigin.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
}
evalSessOrigin

Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh1
found
Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh2
found
Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh4
found
Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8
Restart session found! Take last run
Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh16
Restart session found! Take last run
Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8_AMR1
found
Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8_AMR2
found
Searching for: CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh1
found
Searching for: CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh2
found
Searching for: CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh4
found
Searching for: CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh8
Restart session found! Take last run
Searching for: CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh16
found
Searching for: CapillaryRi

#0: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh1	1/10/2019 9:57:43 AM	f794445a...
#1: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh2	1/14/2019 10:17:04 AM	e2aeae7d...
#2: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh4	1/10/2019 9:57:10 AM	e820b82c...
#3: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8_restart1	1/19/2019 1:56:46 PM	29a22ffc...
#4: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh16_restart2	2/8/2019 6:29:03 PM	45445d6b...
#5: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8_AMR1	4/5/2019 5:40:10 PM	c80abe0f...
#6: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8_AMR2	4/5/2019 5:43:14 PM	bc945e8c...
#7: CapillaryRisePaper	CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh1	11/14/2018 5:31:20 PM	6f2b4cd3...
#8: CapillaryRisePaper	CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh2	5/10/2019 3:54:06 PM	82de42cf...
#9: Cap

In [82]:
NUnit.Framework.Assert.AreEqual(21, evalSessOrigin.Count, $"Not enough sessions for evaluation.");

In [83]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
foreach (string study in studieS) {
foreach (string mesh in mesheS) {

    string studyName = $"CapillaryRise_Omega1_{study}_meshStudy_{mesh}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));

    if (mesh == "mesh1")
        studySess = sessions.Where(sess => sess.Name.Contains(studyName) && !sess.Name.Contains("mesh16"));
    if (mesh == "mesh8")
        studySess = sessions.Where(sess => sess.Name.Contains(studyName) && !sess.Name.Contains("_AMR"));

    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found in wmg! study will be excluded from check against origin session");
        
        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }
        
    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
}
evalSess

Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh1
found
Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh2
found
Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh4
found
Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8
found
Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh16
Restart session found in originDB! Take last run
Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8_AMR1
found in originDB
Searching for: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8_AMR2
found in originDB
Searching for: CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh1
found
Searching for: CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh2
found
Searching for: CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh4
found
Searching for: CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh8
found
Searching for: CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh16
found in originDB
Searching for: CapillaryRise_Omega1_reso

#0: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh1*	12/3/2025 12:08:41 PM	1f63fcdd...
#1: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh2*	12/3/2025 12:09:10 PM	d40aeeff...
#2: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh4*	12/3/2025 12:09:42 PM	6bac848e...
#3: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8*	12/3/2025 12:10:11 PM	70e05c71...
#4: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh16_restart2	2/8/2019 6:29:03 PM	45445d6b...
#5: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8_AMR1	4/5/2019 5:40:10 PM	c80abe0f...
#6: CapillaryRisePaper	CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8_AMR2	4/5/2019 5:43:14 PM	bc945e8c...
#7: CapillaryRisePaper	CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh1*	12/3/2025 12:08:23 PM	2116ae4c...
#8: CapillaryRisePaper	CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh2*	12/3/2025 12:08:52 PM	3ad4f61d...
#9: Ca

In [85]:
NUnit.Framework.Assert.AreEqual(21, evalSess.Count, $"Not enough sessions for evaluation.");

In [ ]:
sessions.AddRange(originDB.Sessions);
sessions

#0: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh1_startUp	11/24/2025 3:07:56 PM	243a4381...
#1: CapillaryRisePaper	CapillaryRise_Omega100_OmegaStudy_mesh8_startUp	11/24/2025 3:09:20 PM	e34da697...
#2: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh2_startUp	11/24/2025 3:08:05 PM	ca56d38d...
#3: CapillaryRisePaper	CapillaryRise_Omega10_OmegaStudy_mesh8_startUp	11/24/2025 3:09:09 PM	6c1e7c74...
#4: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh4_startUp	11/24/2025 3:08:13 PM	0d08da88...
#5: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh8_startUp*	11/24/2025 3:08:23 PM	52e81583...
#6: CapillaryRisePaper	CapillaryRise_Omega05_OmegaStudy_mesh8_AMR1_startUp*	11/24/2025 3:08:51 PM	89838821...
#7: CapillaryRisePaper	CapillaryRise_Omega01_OmegaStudy_mesh8_AMR1_startUp*	12/2/2025 2:06:56 PM	f072f08d...
#8: CapillaryRisePaper	CapillaryRise_Omega1_OmegaStudy_mesh8_startUp	11/24/2025 3:09:00 PM	0e9829d9...
#9: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh16

### compute times

In [18]:
foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = nameData.Length;
    for (int i = 0; i < nameLength; i++) {
        if (nameData[i].StartsWith("restart"))
            continue;
        if (i == 0)
            sessName += nameData[i];
        else    
            sessName += "_" + nameData[i];
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    List<string> dtS = new List<string>();
    dtS.Add(currSI.KeysAndQueries["dtFixed"].ToString());
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        //Console.WriteLine($"   processing session {currSI.Name}");
        var rSI = rSIs.Single();
        if (rSI.Name.ToLower().Contains("_startup"))
            break;
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        dtS.Add(currSI.KeysAndQueries["dtFixed"].ToString());
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    timesteps -= currTimesteps.First().TimeStepNumber.MajorNumber;
    physicalTime -= currTimesteps.First().PhysicalTime;

    //Console.WriteLine($"Session - {sessName}: timesteps = {timesteps}, t_end = {physicalTime} (saveperiod = {currSI.KeysAndQueries["saveperiod"]}, LogPeriod = {currSI.KeysAndQueries["LogPeriod"]}); compute time = {computeTime.Days}:{computeTime.Hours}");
    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps}, t_end = {physicalTime}; compute time = {computeTime.Days}:{computeTime.Hours}");
    
    for (int i = 0; i < dtS.Count; i++) {
        Console.WriteLine($"  dt sess {i}: {dtS[i]}");
    }
}

Session - CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh1: timesteps = 1900, t_end = 0.019000000000019; compute time = 0:0
  dt sess 0: 1E-05
Session - CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh2: timesteps = 38800, t_end = 0.3879999999998994; compute time = 5:21
  dt sess 0: 1E-05
Session - CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh4: timesteps = 27400, t_end = 0.274000000000274; compute time = 5:21
  dt sess 0: 1E-05
Session - CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8: timesteps = 17800, t_end = 0.17800000000017802; compute time = 5:21
  dt sess 0: 1E-05
Session - CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh16: timesteps = 70000, t_end = 0.6999999999985025; compute time = 28:15
  dt sess 0: 1E-05
  dt sess 1: 1E-05
  dt sess 2: 1E-05
Session - CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8_AMR1: timesteps = 70000, t_end = 0.6999999999985026; compute time = 12:16
  dt sess 0: 1E-05
Session - CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8_AMR2: ti

### read log data

In [19]:
public static List<Plot2Ddata>[] ReadMergedDataForMovingContactLine(this IEnumerable<ISessionInfo> sess) {

    List<Plot2Ddata>[] evalData = new List<Plot2Ddata>[2];

    foreach (var evalSess in sess) {
        Console.WriteLine("Processing session: " + evalSess.Name);
        // Get session history
        List<ISessionInfo> restartSessionList = new List<ISessionInfo>();
        restartSessionList.Add(evalSess);
        Guid restartID = evalSess.RestartedFrom;
        while(restartID != Guid.Empty) {
            try {
                var rSess = sessions.Where(sess => sess.ID == restartID).Single();
                if (rSess.Name.ToLower().Contains("_startup"))
                    break;
                Console.WriteLine("  Found restart session: " + rSess.Name);
                restartSessionList.Add(rSess);
                restartID = rSess.RestartedFrom;
            }
            catch { // do nothing if session not found
                restartID = Guid.Empty;
            } 
        }
        restartSessionList.Reverse();

        Console.WriteLine("  Merging data from " + restartSessionList.Count + " sessions.");
        // Read data from all sessions and merge
        var mergedEvalData = restartSessionList.ReadLogDataForMovingContactLine(mergeSess: true);

        for (int i = 0; i < 2; i++) {
            if (evalData[i] == null) {
                evalData[i] = mergedEvalData[i];
            } else {
                for (int j = 0; j < evalData[i].Count(); j++) {
                    evalData[i].ElementAt(j).AddDataGroup(mergedEvalData[i].ElementAt(j).dataGroups.First()); // add first data group (should be only one if mergeSess = true)
                }
            }
        }
    }

    return evalData;
}

In [20]:
var evalConvData = evalSess.ReadMergedDataForMovingContactLine();

Processing session: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh1
  Merging data from 1 sessions.
number of contact lines: 2
Element at 0: time vs contact-pointX
Element at 1: time vs contact-pointY
Element at 2: time vs contact-VelocityX
Element at 3: time vs contact-VelocityY
Element at 4: time vs contact-angle
Processing session: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh2
  Merging data from 1 sessions.
number of contact lines: 2
Element at 0: time vs contact-pointX
Element at 1: time vs contact-pointY
Element at 2: time vs contact-VelocityX
Element at 3: time vs contact-VelocityY
Element at 4: time vs contact-angle
Processing session: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh4
  Merging data from 1 sessions.
number of contact lines: 2
Element at 0: time vs contact-pointX
Element at 1: time vs contact-pointY
Element at 2: time vs contact-VelocityX
Element at 3: time vs contact-VelocityY
Element at 4: time vs contact-angle
Processing session: CapillaryRise_Omega1

In [21]:
var evalConvDataOrigin = evalSessOrigin.ReadMergedDataForMovingContactLine();

Processing session: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh1
  Merging data from 1 sessions.
number of contact lines: 2
Element at 0: time vs contact-pointX
Element at 1: time vs contact-pointY
Element at 2: time vs contact-VelocityX
Element at 3: time vs contact-VelocityY
Element at 4: time vs contact-angle
Processing session: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh2
  Merging data from 1 sessions.
number of contact lines: 2
Element at 0: time vs contact-pointX
Element at 1: time vs contact-pointY
Element at 2: time vs contact-VelocityX
Element at 3: time vs contact-VelocityY
Element at 4: time vs contact-angle
Processing session: CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh4
  Merging data from 1 sessions.
number of contact lines: 2
Element at 0: time vs contact-pointX
Element at 1: time vs contact-pointY
Element at 2: time vs contact-VelocityX
Element at 3: time vs contact-VelocityY
Element at 4: time vs contact-angle
Processing session: CapillaryRise_Omega1

In [22]:
public static List<Plot2Ddata>[] CleanUpDataForMovingContactLine(this List<Plot2Ddata>[] data) {

    List<Plot2Ddata>[] evalData = new List<Plot2Ddata>[2];
    evalData[0] = new List<Plot2Ddata>();   // contact line at symmetry plane
    evalData[1] = new List<Plot2Ddata>();   // contact line at wall 

    int numDataGroups = data[0].ElementAt(0).dataGroups.Count();
    for (int grp = 0; grp < numDataGroups; grp++) {

        //Console.WriteLine("Processing data group: " + data[0].ElementAt(0).dataGroups[grp].Name);

        // get index permutation to sort data according to time
        double[] cLpos = data[0].ElementAt(0).dataGroups[grp].Values;   // check the X-position of the contact line
        int numTimesteps = cLpos.Length;
        int[][] cLIndex = new int[2][];
        cLIndex[0] = new int[numTimesteps];   // contact line at symmetry plane
        cLIndex[1] = new int[numTimesteps];   // contact line at wall
        for (int ts = 0; ts < numTimesteps; ts++) {
            if (cLpos[ts] < 0.0025) {   // contact line at symmetry plane
                cLIndex[0][ts] = 0;
                cLIndex[1][ts] = 1;
            } else {    // contact line at wall
                cLIndex[0][ts] = 1;
                cLIndex[1][ts] = 0;
            }
        }

        // get t0
        double t0 = data[0].ElementAt(0).dataGroups[grp].Abscissas[0];

        // create new data sets
        for (int cL = 0; cL < 2; cL++) {   // cL = 0: contact line at symmetry plane; cL = 1: contact line at wall
            int numData = data[0].Count();
            for (int i = 0; i < numData; i++) {

                //Console.WriteLine($"...  Processing data set at index {i} ...");

                double[] time = new double[numTimesteps];
                double[] value = new double[numTimesteps];

                for (int ts = 0; ts < numTimesteps; ts++) {
                    time[ts] = data[0].ElementAt(i).dataGroups[grp].Abscissas[ts] - t0;
                    value[ts] = data[cLIndex[cL][ts]].ElementAt(i).dataGroups[grp].Values[ts];
                }

                if (evalData[cL].Count() <= i) {
                    Plot2Ddata plotData = new Plot2Ddata();
                    evalData[cL].Add(plotData);
                }   
                evalData[cL].ElementAt(i).AddDataGroup(data[cL].ElementAt(i).dataGroups[grp].Name, time, value);     

                //Console.WriteLine($"... done"); 
            }
        }

    }
    
    return evalData;
}

In [23]:
var cleanEvalConvData = evalConvData.CleanUpDataForMovingContactLine();

In [24]:
var cleanEvalConvDataOrigin = evalConvDataOrigin.CleanUpDataForMovingContactLine();

In [25]:
public static Plot2Ddata GetCapillaryRiseHeightMeshStudy_Plot2Ddata(List<Plot2Ddata>[] evalData, string studyName) {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "t in s";
    plt.Ylabel = "h in mm";

    var datGrpS = evalData[0].ElementAt(1).dataGroups.Where(grp => grp.Name.Contains(studyName));

    int lcIdx = 1;
    foreach (var datGrp in datGrpS) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx);

        double[] values_mm = datGrp.Values.Select(v => v * 1000.0).ToArray(); // convert from m to mm
        string LegendName = "";
        string[] nameData = datGrp.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
        if (nameData.Length == 5) {
            LegendName = nameData[4];
        } else if (nameData.Length == 6) {
            LegendName = nameData[4] + "-" + nameData[5];
        } else {
            throw new ArgumentException("Unexpected number of entries in data group name.");
        }
        plt.AddDataGroup(LegendName, datGrp.Abscissas, values_mm, fmt);

        lcIdx++;
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 0.69;

    plt.ShowLegend = true;

    return plt;        
}

### reference rise height 

In [26]:
static double R = 0.005 * 1000.0; // in mm
static double Theta = 30.0 * Math.PI / 180.0; // in rad
static double riseHeight = R * (4.0 - (2.0 - Math.Sin(Theta) - (Math.Asin(Math.Cos(Theta))/Math.Cos(Theta)))/(2.0 * Math.Cos(Theta))); // in mm

riseHeight

19.160531485066464

In [27]:
Plot2Ddata refPlt = new Plot2Ddata();
refPlt.AddDataGroup("rise height", new double[]{ 0.0, 0.7 }, new double[] { riseHeight, riseHeight }, 
new PlotFormat() { Style = Styles.Lines, LineWidth = 2, LineColor = LineColors.Black });

### Fig. 12 - Mesh study using effective slip

In [28]:
List<Plot2Ddata>[] plotEvalData = (runShortTests) ? cleanEvalConvDataOrigin : cleanEvalConvData;

In [29]:
Plot2Ddata Fig12_a = GetCapillaryRiseHeightMeshStudy_Plot2Ddata(plotEvalData, "effectiveSlip");
Fig12_a.AddDataGroup(refPlt.dataGroups[0]);
Fig12_a.LegendAlignment = new string[] { "o", "t", "r" };

Fig12_a.ToGnuplot().PlotSVG(xRes:800,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 10 
 
 
 
 
 12 
 
 
 
 
 14 
 
 
 
 
 16 
 
 
 
 
 18 
 
 
 
 
 20 
 
 
 
 
 22 
 
 
 
 
 24 
 
 
 
 
 26 
 
 
 
 
 28 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 h in mm 
 
 
 
 
 t in s 
 
 
 
 
 mesh1 
 
 
 
 
 mesh1 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M605.0,48.1 L658.4,48.1 M63.6,416.6 L63.6,417.0 L64.3,416.9 L65.1,416.7 L65.8,416.4 L66.5,415.9
 L67.2,415.4 L67.9,414.7 L68.6,414.0 L69.3,413.2 L70.0,412.3 L70.7,411.4 L71.4,410.4 L72.1,409.4
 L72.8,408.3 L73.6,407.1 L74.3,405.9 L75.0,404.6 L75.7,403.3 L76.4,402.0 L77.1,400.7 L77.8,399.3
 L78.5,397.9 L79.2,396.5 L79.9,394.9 L80.6,393.3 L81.3,391.6 L82.1,389.8 L82.8,387.9 L83.5,386.0
 L84.2,384.0 L84.9,381.9 L85.6,379.7 L86.3,377.5 L87.0,375.2 L87.7,372.9 L88.4,370.5 L89.1,368.1
 L89.8,365.7 L90.6,363.1 L91.3,360.6 L92.0,358.0 L92.7,355.4 L93.4,352.7 L94.1,350.1 L94.8,347.5
 L95.5,344.9 L96.2,342.3 L96.9,339.6 L97.6,337.0 L98.3,334.4 L99.1,331.7 L99.8,329.1 L100.5,326.5
 L101.2,323.8 L101.9,321.1 L102.6,318.3 L103.3,315.5 L104.0,312.7 L104.7,309.8 L105.4,306.9 L106.1,303.9
 L106.8,300.9 L107.6,298.1 L108.3,295.3 L109.0,292.6 L109.7,289.8 L110.4,287.0 L111.1,284.1 L111.8,281.2
 L112.5,278.3 L113.2,275.3 L113.9,272.3 L114.6,269.2 L115.3,266.2 L116.1,263.1 L116.8,260.0 L117.5,256.9
 L118.2,253.8 L118.9,250.7 L119.6,247.7 L120.3,244.7 L121.0,241.8 L121.7,238.9 L122.4,236.0 L123.1,233.2
 L123.8,230.3 L124.6,227.6 L125.3,224.8 L126.0,222.1 L126.7,219.3 L127.4,216.5 L128.1,213.7 L128.8,210.9
 L129.5,208.1 L130.2,205.3 L130.9,202.5 L131.6,199.6 L132.4,197.0 L133.1,194.6 L133.8,192.1 L134.5,189.7
 L135.2,187.2 L135.9,184.6 L136.6,182.1 L137.3,179.5 L138.0,176.9 L138.7,174.2 L139.4,171.6 L140.1,168.9
 L140.9,166.3 L141.6,163.6 L142.3,161.0 L143.0,158.4 L143.7,155.8 L144.4,153.3 L145.1,150.8 L145.8,148.3
 L146.5,145.9 L147.2,143.6 L147.9,141.3 L148.6,139.0 L149.4,136.8 L150.1,134.7 L150.8,132.6 L151.5,130.5
 L152.2,128.4 L152.9,126.4 L153.6,124.3 L154.3,122.3 L155.0,120.3 L155.7,118.3 L156.4,116.3 L157.1,114.3
 L157.9,112.3 L158.6,110.3 L159.3,108.4 L160.0,106.4 L160.7,104.4 L161.4,102.5 L162.1,100.6 L162.8,98.7
 L163.5,96.9 L164.2,95.3 L164.9,93.8 L165.6,92.3 L166.4,90.8 L167.1,89.3 L167.8,87.8 L168.5,86.3
 L169.2,84.8 L169.9,83.3 L170.6,81.9 L171.3,80.4 L172.0,79.0 L172.7,77.5 L173.4,76.1 L174.1,74.7
 L174.9,73.3 L175.6,72.0 L176.3,70.7 L177.0,69.4 L177.7,68.2 L178.4,66.9 L179.1,65.8 L179.8,64.6
 L180.5,63.5 L181.2,62.4 L181.9,61.3 L182.6,60.3 L183.4,59.3 L184.1,58.3 L184.8,57.4 L185.5,56.5
 L186.2,55.6 L186.9,54.7 L187.6,53.9 L188.3,53.1 L189.0,52.3 L189.7,51.5 L190.4,50.8 L191.1,50.1
 L191.9,49.5 L192.6,48.8 L193.3,48.2 L194.0,47.6 L194.7,47.0 L195.4,46.5 L196.1,46.0 L196.8,45.5
 L197.5,45.0 L198.2,44.6 L198.9,44.2 L199.6,43.9 L200.4,43.5 L201.1,43.2 L201.8,42.9 L202.5,42.7
 L203.2,42.4 L203.9,42.2 L204.6,42.1 L205.3,41.9 L206.0,41.8 L206.7,41.7 L207.4,41.6 L208.2,41.5
 L208.9,41.5 L209.6,41.5 L210.3,41.5 L211.0,41.6 L211.7,41.6 L212.4,41.7 L213.1,41.9 L213.8,42.0
 L214.5,42.2 L215.2,42.4 L215.9,42.6 L216.7,42.8 L217.4,43.1 L218.1,43.4 L218.8,43.7 L219.5,44.1
 L220.2,44.5 L220.9,44.9 L221.6,45.3 L222.3,45.7 L223.0,46.2 L223.7,46.7 L224.4,47.2 L225.2,47.7
 L225.9,48.3 L226.6,48.9 L227.3,49.5 L228.0,50.1 L228.7,50.8 L229.4,51.4 L230.1,52.1 L230.8,52.9
 L231.5,53.6 L232.2,54.3 L232.9,55.1 L233.7,55.9 L234.4,56.7 L235.1,57.5 L235.8,58.4 L236.5,59.3
 L237.2,60.2 L237.9,61.1 L238.6,62.0 L239.3,62.9 L240.0,63.9 L240.7,64.8 L241.4,65.8 L242.2,66.8


In [64]:
public static void CheckAgainstOriginData(List<Plot2Ddata>[] StudyData, List<Plot2Ddata>[] OriginData, string study, double errThrsh, bool output = false) {

    
    var StudyGrps = StudyData[0].ElementAt(1).dataGroups.Where(grp => grp.Name.Contains($"{study}_meshStudy_"));

    foreach (var Sgrp in StudyGrps) {

        if (Sgrp.Name.Contains("mesh1") && !Sgrp.Name.Contains("mesh16"))
            continue;

        if (output) {
            var nameData = Sgrp.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
            if (nameData.Length == 6) {
                Console.WriteLine($"checking {nameData[4]}_{nameData[5]}:"); 
            } else {
                Console.WriteLine($"checking {nameData[4]}:"); 
            }  
        } 

        var Ogrp = OriginData[0].ElementAt(1).dataGroups.Where(grp => grp.Name == Sgrp.Name).Single();

        var errorData = ISessionInfoExtensions.ComputeErrorNormsForLogData(Sgrp, Ogrp);

        if (output) {
            Console.WriteLine($"    l1-error norm = {errorData["l_1 error norm"]}");
            Console.WriteLine($"    l2-error norm = {errorData["l_2 error norm"]}");
            Console.WriteLine($"    linf-error norm = {errorData["l_inf error norm"]}");
        }

        
        NUnit.Framework.Assert.LessOrEqual(errorData["l_1 error norm"], errThrsh,  
            $"l_1 error norm larger than allowed: {errorData["l_1 error norm"]}");

        NUnit.Framework.Assert.LessOrEqual(errorData["l_2 error norm"], errThrsh,  
            $"l_2 error norm larger than allowed: {errorData["l_2 error norm"]}");

        NUnit.Framework.Assert.LessOrEqual(errorData["l_inf error norm"], errThrsh,  
            $"l_inf error norm larger than allowed: {errorData["l_inf error norm"]}");
    }

}

In [67]:
CheckAgainstOriginData(cleanEvalConvData, cleanEvalConvDataOrigin, "effectiveSlip", 0.01, false);

### Fig. 12 - Mesh study using resolved slip with $L = R/5$

In [33]:
Plot2Ddata Fig12_b = GetCapillaryRiseHeightMeshStudy_Plot2Ddata(plotEvalData, "resolvedSlipR5_");
Fig12_b.AddDataGroup(refPlt.dataGroups[0]);
Fig12_b.LegendAlignment = new string[] { "o", "t", "r" };

Fig12_b.ToGnuplot().PlotSVG(xRes:800,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 10 
 
 
 
 
 12 
 
 
 
 
 14 
 
 
 
 
 16 
 
 
 
 
 18 
 
 
 
 
 20 
 
 
 
 
 22 
 
 
 
 
 24 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 h in mm 
 
 
 
 
 t in s 
 
 
 
 
 mesh1 
 
 
 
 
 mesh1 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M605.0,48.1 L658.4,48.1 M63.6,410.2 L63.6,410.6 L64.3,410.5 L65.1,410.2 L65.8,409.8 L66.5,409.3
 L67.2,408.6 L67.9,407.8 L68.6,406.9 L69.3,405.9 L70.0,404.8 L70.7,403.7 L71.4,402.5 L72.1,401.2
 L72.8,399.8 L73.6,398.4 L74.3,396.9 L75.0,395.3 L75.7,393.7 L76.4,392.0 L77.1,390.4 L77.8,388.8
 L78.5,387.1 L79.2,385.4 L79.9,383.7 L80.6,381.8 L81.3,379.9 L82.1,378.0 L82.8,375.9 L83.5,373.8
 L84.2,371.7 L84.9,369.4 L85.6,367.1 L86.3,364.8 L87.0,362.4 L87.7,359.9 L88.4,357.4 L89.1,354.8
 L89.8,352.3 L90.6,349.7 L91.3,347.1 L92.0,344.4 L92.7,341.6 L93.4,338.9 L94.1,336.1 L94.8,333.4
 L95.5,330.6 L96.2,327.9 L96.9,325.2 L97.6,322.5 L98.3,319.8 L99.1,317.1 L99.8,314.4 L100.5,311.7
 L101.2,309.0 L101.9,306.3 L102.6,303.7 L103.3,301.0 L104.0,298.4 L104.7,295.8 L105.4,293.1 L106.1,290.5
 L106.8,287.8 L107.6,285.0 L108.3,282.2 L109.0,279.4 L109.7,276.6 L110.4,273.7 L111.1,270.9 L111.8,268.1
 L112.5,265.3 L113.2,262.8 L113.9,260.5 L114.6,258.1 L115.3,255.7 L116.1,253.3 L116.8,250.9 L117.5,248.4
 L118.2,246.0 L118.9,243.5 L119.6,240.9 L120.3,238.4 L121.0,235.8 L121.7,233.2 L122.4,230.6 L123.1,227.9
 L123.8,225.3 L124.6,222.7 L125.3,220.1 L126.0,217.5 L126.7,214.9 L127.4,212.3 L128.1,209.7 L128.8,207.2
 L129.5,204.7 L130.2,202.3 L130.9,200.0 L131.6,197.6 L132.4,195.3 L133.1,193.1 L133.8,190.9 L134.5,188.7
 L135.2,186.5 L135.9,184.4 L136.6,182.3 L137.3,180.3 L138.0,178.2 L138.7,176.2 L139.4,174.3 L140.1,172.3
 L140.9,170.4 L141.6,168.4 L142.3,166.6 L143.0,164.7 L143.7,162.8 L144.4,160.8 L145.1,158.9 L145.8,156.9
 L146.5,155.0 L147.2,153.1 L147.9,151.2 L148.6,149.4 L149.4,147.5 L150.1,145.7 L150.8,143.9 L151.5,142.3
 L152.2,140.7 L152.9,139.0 L153.6,137.5 L154.3,136.2 L155.0,135.0 L155.7,133.8 L156.4,132.6 L157.1,131.4
 L157.9,130.1 L158.6,128.9 L159.3,127.6 L160.0,126.4 L160.7,125.1 L161.4,123.8 L162.1,122.6 L162.8,121.3
 L163.5,120.1 L164.2,118.8 L164.9,117.6 L165.6,116.4 L166.4,115.2 L167.1,114.0 L167.8,112.9 L168.5,111.8
 L169.2,110.7 L169.9,109.6 L170.6,108.6 L171.3,107.5 L172.0,106.6 L172.7,105.6 L173.4,104.6 L174.1,103.7
 L174.9,102.8 L175.6,101.9 L176.3,101.0 L177.0,100.2 L177.7,99.4 L178.4,98.6 L179.1,97.8 L179.8,97.0
 L180.5,96.3 L181.2,95.5 L181.9,94.8 L182.6,94.1 L183.4,93.4 L184.1,92.8 L184.8,92.1 L185.5,91.5
 L186.2,90.9 L186.9,90.3 L187.6,89.7 L188.3,89.2 L189.0,88.7 L189.7,88.2 L190.4,87.7 L191.1,87.3
 L191.9,86.8 L192.6,86.4 L193.3,86.0 L194.0,85.7 L194.7,85.3 L195.4,85.0 L196.1,84.7 L196.8,84.4
 L197.5,84.1 L198.2,83.8 L198.9,83.6 L199.6,83.3 L200.4,83.1 L201.1,82.9 L201.8,82.8 L202.5,82.6
 L203.2,82.4 L203.9,82.3 L204.6,82.2 L205.3,82.1 L206.0,82.0 L206.7,81.9 L207.4,81.9 L208.2,81.8
 L208.9,81.8 L209.6,81.8 L210.3,81.8 L211.0,81.9 L211.7,81.9 L212.4,82.0 L213.1,82.0 L213.8,82.1
 L214.5,82.2 L215.2,82.3 L215.9,82.5 L216.7,82.6 L217.4,82.8 L218.1,82.9 L218.8,83.1 L219.5,83.3
 L220.2,83.5 L220.9,83.7 L221.6,84.0 L222.3,84.2 L223.0,84.5 L223.7,84.7 L224.4,85.0 L225.2,85.3
 L225.9,85.6 L226.6,85.9 L227.3,86.2 L228.0,86.5 L228.7,86.8 L229.4,87.2 L230.1,87.5 L230.8,87.9
 L231.5,88.3 L232.2,88.7 L232.9,89.0 L233.7,89.4 L234.4,89.9 L235.1,90.3 L235.8,90.7 L236.5,91.1
 L237.2,91.6 L237.9,92.0 L238.6,92.5 L239.3,92.9 L240.0,93.4 L240.7,93.9 L241.4,94.3 L242.2,94.8
 L242.9,95.3 L243.6,95.8 

In [69]:
CheckAgainstOriginData(cleanEvalConvData, cleanEvalConvDataOrigin, "resolvedSlipR5", 0.02, false);

### Fig. 14 - Convergence plot using resolved slip 

In [35]:
public static Plot2Ddata GetNormalizedMaxDifferenceForRiseHeigthConvergence_Plot2Ddata(Plot2Ddata data, string study, double[] meshes, string finestSol) {

    List<Plot2Ddata> studyData = new List<Plot2Ddata>();  

    Plot2Ddata plt = new Plot2Ddata();
    var dgrp = data.dataGroups.Where(grp => grp.Name.Contains($"{study}_meshStudy_{finestSol}")).Single();
    plt.AddDataGroup(dgrp);
    studyData.Add(plt);

    for (int i = 0; i < meshes.Length; i++) {
        try {
            Console.WriteLine($"Searching for data group: {study}_meshStudy_mesh{(int)meshes[i]}");
            if (meshes[i] == 1)
                dgrp = data.dataGroups.Where(grp => grp.Name.Contains($"{study}_meshStudy_mesh{(int)meshes[i]}") && !grp.Name.Contains("mesh16")).Single();
            else if (meshes[i] == 8)
                dgrp = data.dataGroups.Where(grp => grp.Name.Contains($"{study}_meshStudy_mesh{(int)meshes[i]}") && !grp.Name.Contains("_AMR")).Single();
            // else if (meshes[i] == 16)
            //     dgrp = data.dataGroups.Where(grp => grp.Name.Contains($"{study}_meshStudy_mesh8_AMR1")).Single();
            else
                dgrp = data.dataGroups.Where(grp => grp.Name.Contains($"{study}_meshStudy_mesh{(int)meshes[i]}")).Single();
        } catch {
            Console.WriteLine($"Data for mesh{(int)meshes[i]} not found!");
            continue;
        }
        plt.AddDataGroup(dgrp);
    }
    studyData.Add(plt);

    var convData = ISessionInfoExtensions.LogDataToConvergenceData(studyData, meshes);

    Plot2Ddata retPlt = new Plot2Ddata().WithLogX().WithLogY();
    retPlt.Xlabel = "cells/R";
    retPlt.Ylabel = "normalized max difference";

    retPlt.AddDataGroup(convData[0].dataGroups[2]);

    return retPlt;
}

In [37]:
Plot2Ddata[,] Fig14 = new Plot2Ddata[1, 2];

Fig14[0, 0] = GetNormalizedMaxDifferenceForRiseHeigthConvergence_Plot2Ddata(plotEvalData[0].ElementAt(1), "resolvedSlipR5", new double[] { 16, 8, 4, 2, 1}, "mesh8_AMR2");

Fig14[0, 1] = GetNormalizedMaxDifferenceForRiseHeigthConvergence_Plot2Ddata(plotEvalData[0].ElementAt(1), "resolvedSlipR50", new double[] { 16, 8, 4, 2}, "mesh8_AMR2");

Fig14.ToGnuplot().PlotSVG(xRes:1200,yRes:500)

Searching for data group: resolvedSlipR5_meshStudy_mesh16
Searching for data group: resolvedSlipR5_meshStudy_mesh8
Searching for data group: resolvedSlipR5_meshStudy_mesh4
Searching for data group: resolvedSlipR5_meshStudy_mesh2
Searching for data group: resolvedSlipR5_meshStudy_mesh1
Searching for data group: resolvedSlipR50_meshStudy_mesh16
Searching for data group: resolvedSlipR50_meshStudy_mesh8
Searching for data group: resolvedSlipR50_meshStudy_mesh4
Searching for data group: resolvedSlipR50_meshStudy_mesh2
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 10 1 
 
 
 
 
 10 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 normalized max difference 
 
 
 
 
 cells/R 
 
 
 
 
 linf error norm 
 
 
 l inf error norm 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -3 
 
 
 
	<path stroke='black' d='M669.5,377.8 L674.0,377.8 M1144.1,377.8 L1139.6,377.8 M669.5,342.9 L674.0,342.9 M1144.1,342.9 L1139.6,342.9
 M669.5,318.2 L674.0,318.2 M1144.1,318.2 L1139.6,318.2 M669.5,299.0 L674.0,299.0 M1144.1,299.0 L1139.6,299.0
 M669.5,283.3 L674.0,283.3 M1144.1,283.3 L1139.6,283.3 M669.5,270.0 L674.0,270.0 M1144.1,270.0 L1139.6,270.0
 M669.5,258.5 L674.0,258.5 M1144.1,258.5 L1139.6,258.5 M669.5,248.4 L674.0,248.4 M1144.1,248.4 L1139.6,248.4
 M669.5,239.3 L678.5,239.3 M1144.1,239.3 L1135.1,239.3 '/> 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 10 1 
 
 
 
	<path stroke='black' d='M978.2,437.5 L978.2,433.0 M978.2,41.1 L978.2,45.6 M1020.0,437.5 L1020.0,433.0 M1020.0,41.1 L1020.0,45.6
 M1049.7,437.5 L1049.7,433.0 M1049.7,41.1 L1049.7,45.6 M1072.7,437.5 L1072.7,433.0 M1072.7,41.1 L1072.7,45.6
 M1091.5,437.5 L1091.5,433.0 M1091.5,41.1 L1091.5,45.6 M1107.3,437.5 L1107.3,433.0 M1107.3,41.1 L1107.3,45.6
 M1121.1,437.5 L1121.1,433.0 M1121.1,41.1 L1121.1,45.6 M1133.2,437.5 L1133.2,433.0 M1133.2,41.1 L1133.2,45.6
 M1144.1,437.5 L1144.1,428.5 M1144.1,41.1 L1144.1,50.1 '/> 
 10 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 normalized max difference 
 
 
 
 
 cells/R 
 
 
 
 
 linf error norm 
 
 
 l inf error norm

In [38]:
double slope = Math.Abs(Fig14[0,0].Regression().ElementAt(0).Value);

NUnit.Framework.Assert.Greater(slope, 1.0,  
    $"regression slope for resolved slip with L = R/5 is too small: {slope}");

In [39]:
double slope = Math.Abs(Fig14[0,1].Regression().ElementAt(0).Value);

NUnit.Framework.Assert.Greater(slope, 1.4,  
    $"regression slope for resolved slip with L = R/50 is too small: {slope}");

### Fig. 19 (Appendix A1) - Mesh study using resolved slip with $L = R/50$

In [ ]:
Plot2Ddata Fig19 = GetCapillaryRiseHeightMeshStudy_Plot2Ddata(plotEvalData, "resolvedSlipR50_");
Fig19.AddDataGroup(refPlt.dataGroups[0]);
Fig19.LegendAlignment = new string[] { "o", "t", "r" };

Fig19.ToGnuplot().PlotSVG(xRes:800,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 10 
 
 
 
 
 12 
 
 
 
 
 14 
 
 
 
 
 16 
 
 
 
 
 18 
 
 
 
 
 20 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 h in mm 
 
 
 
 
 t in s 
 
 
 
 
 mesh1 
 
 
 
 
 mesh1 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M605.0,48.1 L658.4,48.1 M63.6,399.4 L63.6,399.8 L64.3,402.4 L65.1,401.9 L65.8,401.4 L66.5,400.4
 L67.2,399.2 L67.9,398.0 L68.6,396.5 L69.3,395.0 L70.0,393.3 L70.7,391.5 L71.4,389.6 L72.1,387.6
 L72.8,385.6 L73.6,383.4 L74.3,381.0 L75.0,379.2 L75.7,376.8 L76.4,374.8 L77.1,372.2 L77.8,369.5
 L78.5,367.0 L79.2,364.4 L79.9,361.5 L80.6,359.0 L81.3,356.5 L82.1,353.6 L82.8,351.1 L83.5,348.3
 L84.2,345.8 L84.9,343.3 L85.6,340.5 L86.3,338.0 L87.0,335.3 L87.7,332.7 L88.4,330.2 L89.1,327.5
 L89.8,325.0 L90.6,322.3 L91.3,319.8 L92.0,317.3 L92.7,314.8 L93.4,312.3 L94.1,309.9 L94.8,307.5
 L95.5,305.1 L96.2,302.8 L96.9,300.5 L97.6,298.3 L98.3,296.1 L99.1,294.0 L99.8,292.0 L100.5,290.0
 L101.2,288.1 L101.9,286.2 L102.6,284.3 L103.3,282.6 L104.0,280.8 L104.7,279.1 L105.4,277.5 L106.1,275.8
 L106.8,274.3 L107.6,272.7 L108.3,271.2 L109.0,269.7 L109.7,268.3 L110.4,266.9 L111.1,265.5 L111.8,264.2
 L112.5,262.8 L113.2,261.5 L113.9,260.2 L114.6,259.0 L115.3,256.4 L116.1,255.3 L116.8,254.4 L117.5,253.4
 L118.2,252.4 L118.9,251.7 L119.6,250.7 L120.3,249.7 L121.0,249.0 L121.7,248.1 L122.4,247.4 L123.1,246.5
 L123.8,245.6 L124.6,245.0 L125.3,244.1 L126.0,243.5 L126.7,242.7 L127.4,241.9 L128.1,241.4 L128.8,240.6
 L129.5,240.1 L130.2,239.3 L130.9,238.6 L131.6,238.0 L132.4,237.0 L133.1,236.1 L133.8,235.0 L134.5,234.0
 L135.2,233.1 L135.9,232.1 L136.6,231.3 L137.3,230.4 L138.0,229.4 L138.7,228.6 L139.4,227.7 L140.1,226.9
 L140.9,226.1 L141.6,225.2 L142.3,224.4 L143.0,223.6 L143.7,222.8 L144.4,222.0 L145.1,221.1 L145.8,220.4
 L146.5,219.5 L147.2,218.8 L147.9,218.0 L148.6,217.2 L149.4,216.5 L150.1,215.6 L150.8,215.0 L151.5,214.2
 L152.2,213.3 L152.9,212.7 L153.6,211.9 L154.3,211.2 L155.0,210.4 L155.7,209.7 L156.4,209.4 L157.1,208.6
 L157.9,208.9 L158.6,208.0 L159.3,207.1 L160.0,207.2 L160.7,206.2 L161.4,206.1 L162.1,205.1 L162.8,204.1
 L163.5,203.3 L164.2,202.4 L164.9,201.3 L165.6,200.5 L166.4,199.7 L167.1,198.7 L167.8,198.1 L168.5,197.1
 L169.2,196.5 L169.9,195.8 L170.6,194.9 L171.3,194.3 L172.0,193.4 L172.7,192.8 L173.4,192.1 L174.1,191.2
 L174.9,190.6 L175.6,189.7 L176.3,189.0 L177.0,188.3 L177.7,187.4 L178.4,186.7 L179.1,185.8 L179.8,185.0
 L180.5,184.3 L181.2,183.4 L181.9,182.7 L182.6,181.8 L183.4,181.0 L184.1,180.2 L184.8,179.4 L185.5,178.6
 L186.2,177.7 L186.9,177.0 L187.6,176.2 L188.3,175.4 L189.0,174.6 L189.7,173.8 L190.4,173.0 L191.1,172.3
 L191.9,171.5 L192.6,170.8 L193.3,170.0 L194.0,169.2 L194.7,168.5 L195.4,167.8 L196.1,167.0 L196.8,166.3
 L197.5,165.6 L198.2,164.9 L198.9,164.2 L199.6,163.5 L200.4,162.8 L201.1,162.1 L201.8,161.4 L202.5,160.7
 L203.2,160.0 L203.9,159.4 L204.6,158.7 L205.3,158.1 L206.0,157.4 L206.7,156.8 L207.4,156.1 L208.2,155.5
 L208.9,154.9 L209.6,154.3 L210.3,153.7 L211.0,153.1 L211.7,152.5 L212.4,151.9 L213.1,151.3 L213.8,150.7
 L214.5,150.2 L215.2,149.6 L215.9,149.1 L216.7,148.5 L217.4,148.0 L218.1,147.5 L218.8,146.9 L219.5,146.4
 L220.2,145.9 L220.9,145.4 L221.6,144.9 L222.3,144.4 L223.0,143.8 L223.7,143.4 L224.4,142.9 L225.2,142.4
 L225.9,141.9 L226.6,141.4 L227.3,141.0 L228.0,140.5 L228.7,140.1 L229.4,139.6 L230.1,139.2 L230.8,138.8
 L231.5,138.3 L232.2,137.9 L232.9,137.5 L233.7,137.0 L234.4,136.7 L235.1,136.2 L235.8,135.9 L236.5,135.4
 L237.2,135.0 L237.9,134.7 L238.6,134.3 L239.3,133.9 L240.0,133.6 L240.7,133.2 L241.

In [71]:
CheckAgainstOriginData(cleanEvalConvData, cleanEvalConvDataOrigin, "resolvedSlipR50", 0.03, false);